In [1]:
# single cell libraries
import scanpy as sc
import anndata as ad
import scrublet as scr
import leidenalg, igraph
import bbknn

# processing tools
import numpy as np
import pandas as pd
import scipy
import itertools
from fa2 import ForceAtlas2

# plotting tools
import matplotlib.pyplot as plt

# file grab
from pathlib import Path
import pooch
import hashlib

/root/miniconda/envs/unified_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/root/miniconda/envs/unified_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sc.settings.verbosity = 2
sc.logging.print_versions()
results_file = "./write/_"

sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(3, 3), facecolor="white")

In [3]:
# restore working adata
tcell = sc.read_h5ad("cache/tcell_pbmc68k.h5ad")
tcell

AnnData object with n_obs × n_vars = 42168 × 12792
    obs: 'celltype', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'leiden_r0.5', 'leiden_r0.8', 'leiden_r1.0', 'leiden_r1.2', 'leiden_res1.0', 'leiden_res1.2', 'leiden_res1.5', 'leiden_res0.8', 'leiden_res0.5'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'diffmap_evals', 'draw_graph', 'hvg', 'leiden_r0.5', 

In [9]:
# checking T cell and NK cell markers before subsetting

sc.pl.umap(
    adata,
    color=["CD3D","CD3E","CD3G", "NCAM1", "KLRD1"],
    cmap="viridis",
    vmin=0, vmax="p99",
    save="_4_tcell_nk_markers.png",
)

In [39]:
tcell = adata[adata.obs['leiden_r1.0'].isin(
    ['0', '1', '2', '3', '4', '5', '6']
)].copy()

NameError: name 'adata' is not defined

In [15]:
sc.tl.diffmap(tcell)
sc.pp.neighbors(tcell, n_neighbors=10, use_rep="X_diffmap")

computing Diffusion Maps using n_comps=15(=n_dcs)
computing transitions
    finished (0:00:00)
    eigenvalues of transition matrix
    [1.         0.9605462  0.8275097  0.8198901  0.8042307  0.7983387
     0.7945762  0.79339105 0.791062   0.7892527  0.7874667  0.7846262
     0.7834075  0.78208923 0.7775119 ]
    finished (0:00:04)
computing neighbors
    finished (0:00:22)


In [16]:
sc.tl.draw_graph(tcell)
sc.pl.draw_graph(
    tcell, color="leiden_r1.0", legend_loc="on data", layout='fa',
    save="_tcell_r1.0.png",
)

drawing single-cell graph using layout 'fa'
    finished (0:06:14)


In [26]:
# paga - adata 1
sc.tl.paga(tcell, groups="leiden_r1.0")
sc.pl.paga(
    tcell, color=["leiden_r1.0", "CCR7", "PTPRC", "IL7R"], save="_tcell_markers_1.png"
)

running PAGA
    finished (0:00:00)


In [27]:
# paga - adata 2
sc.tl.paga(tcell, groups="leiden_r1.0")
sc.pl.paga(
    tcell, color=["GZMK", "GZMB", "FOXP3"], save="_tcell_markers_2.png"
)

running PAGA
    finished (0:00:00)


In [4]:
# leiden again for new tcell

sc.tl.leiden(tcell, flavor="igraph", resolution=0.3, key_added="leiden_res0.3", random_state=0)
sc.tl.leiden(tcell, flavor="igraph", resolution=0.5, key_added="leiden_res0.5", random_state=0)
sc.tl.leiden(tcell, flavor="igraph", resolution=0.8, key_added="leiden_res0.8", random_state=0)
tcell

running Leiden clustering
    finished (0:00:00)
running Leiden clustering
    finished (0:00:00)
running Leiden clustering
    finished (0:00:00)


AnnData object with n_obs × n_vars = 42168 × 12792
    obs: 'celltype', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'leiden_r0.5', 'leiden_r0.8', 'leiden_r1.0', 'leiden_r1.2', 'leiden_res1.0', 'leiden_res1.2', 'leiden_res1.5', 'leiden_res0.8', 'leiden_res0.5', 'leiden_res0.3'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'diffmap_evals', 'draw_graph', 'hvg'

In [7]:
# umap for leiden_res0.3
sc.pl.umap(tcell, color=["leiden_res0.3", "leiden_res0.5", "leiden_res0.8"],
           save="_5_tcellResOptim.png")

In [5]:
# draw_graph for leiden_res0.3
## sc.tl.draw_graph(tcell)
sc.pl.draw_graph(
    tcell, color="leiden_res0.3", legend_loc="on data", layout='fa',
    save="_tcell2_res0.3.png",
)

In [33]:
# paga to filter NK versus NKT
colors = ["leiden_res0.3","TRAC","CCR7","SELL","IL7R","TCF7","LEF1",
           "CD8A","GZMK","GZMB","PRF1","IFNG","NKG7","GNLY",
           "CD4","FOXP3","IL2RA","CTLA4","IKZF2","KLRD1","NCAM1"]

for k in colors:
    if k in tcell.obs and pd.api.types.is_categorical_dtype(tcell.obs[k].dtype):
        tcell.obs[k] = tcell.obs[k].cat.remove_unused_categories()
        tcell.uns.pop(f"{k}_colors", None)
        
# plot in 3 x 7 grid
ncols, nrows = 3, 7
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.0, nrows*3.0))
axes = axes.ravel()

for i, k in enumerate(colors):
    pre = set(fig.axes)
    sc.pl.paga(tcell, color=k, threshold=0.05, ax=axes[i], title=k, show=False)
    new_axes = [a for a in fig.axes if a not in pre and a is not axes[i]]
    for a in new_axes: a.remove()

plt.tight_layout()
plt.savefig("./figures/paga_3x7.png", dpi=200)

/tmp/ipykernel_432/3000630691.py:6: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if k in tcell.obs and pd.api.types.is_categorical_dtype(tcell.obs[k].dtype):


In [34]:
# found out cluster 8 was NK, drop NK 
tcell = tcell[~tcell.obs['leiden_res0.3'].isin(['8'])].copy()
tcell.obs['leiden_res0.3'] = (
    tcell.obs['leiden_res0.3']
    .cat.remove_unused_categories()
)
tcell

AnnData object with n_obs × n_vars = 36625 × 12792
    obs: 'celltype', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'leiden_r0.5', 'leiden_r0.8', 'leiden_r1.0', 'leiden_r1.2', 'leiden_res1.0', 'leiden_res1.2', 'leiden_res1.5', 'leiden_res0.8', 'leiden_res0.5', 'leiden_res0.3'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'diffmap_evals', 'draw_graph', 'hvg'

In [43]:
# recompute neighbors, leiden, check in umap
##sc.tl.diffmap(tcell)
##sc.pp.neighbors(tcell, n_neighbors=10, use_rep="X_diffmap")
##sc.tl.leiden(tcell, flavor="igraph", resolution=1.0, key_added="leiden_res1.0_2")
##sc.tl.leiden(tcell, flavor="igraph", resolution=0.5, key_added="leiden_res0.5_2")
##sc.tl.leiden(tcell, flavor="igraph", resolution=0.8, key_added="leiden_res0.8_2")
##sc.tl.umap(tcell)
sc.pl.umap(tcell, color=["leiden_res0.5_2", "leiden_res0.8_2", "leiden_res1.0_2"], save="_tcell4_leidenres_2.png")

In [57]:
# recompute paga and check in grid
sc.tl.paga(tcell, groups="leiden_res0.5_2")

markers = ["leiden_res0.5_2","CD3D", "CD3E", "CD3G", "TRAC","CCR7","SELL","IL7R","TCF7","LEF1",
           "CD8A","GZMK","GZMB","PRF1","IFNG","NKG7","GNLY",
           "CD4","FOXP3","IL2RA","CTLA4","IKZF2","KLRD1","NCAM1"]

for k in markers:
    if k in tcell.obs and pd.api.types.is_categorical_dtype(tcell.obs[k].dtype):
        tcell.obs[k] = tcell.obs[k].cat.remove_unused_categories()
        tcell.uns.pop(f"{k}_colors", None)
        
# plot in 3 x 8 grid
ncols, nrows = 3, 8
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.0, nrows*3.0))
axes = axes.ravel()

for i, k in enumerate(markers):
    pre = set(fig.axes)
    sc.pl.paga(tcell, color=k, threshold=0.1, ax=axes[i], title=k, show=False)
    new_axes = [a for a in fig.axes if a not in pre and a is not axes[i]]
    for a in new_axes: a.remove()

plt.tight_layout()
plt.savefig("./figures/paga_2_3X8.png", dpi=200)

running PAGA
    finished (0:00:00)


/tmp/ipykernel_432/4245388225.py:9: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if k in tcell.obs and pd.api.types.is_categorical_dtype(tcell.obs[k].dtype):


In [31]:
tcell.write("cache/tcell_pbmc68k.h5ad")

In [49]:
"TRBC" in tcell.var_names

False

In [30]:

sc.__version__

'1.11.4'